<a href="https://colab.research.google.com/github/neoaitech/Neoai_attendance_system/blob/feature%2Fpriti-face-detection/face_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install face_recognition


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=0f84ca05fe6171fe1aacf8f923fcb56aef2d0c8dfaf7ccc02b7cb8d5f36e939a
  Stored in directory: /root/.cache/pip/wheels/39/20/47/6b647a3457f0e36c161b2516a5bd7f29f6bbd5e3a67c46a710
Successfully built face-recognition-models


In [ ]:
from sklearn.datasets import fetch_lfw_people

# Load dataset with people who have multiple photos
lfw = fetch_lfw_people(min_faces_per_person=2, resize=0.5)

print(f"Total images: {lfw.images.shape[0]}")
print(f"People: {len(lfw.target_names)}")

Total images: 9164
People: 1680


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from sklearn.datasets import fetch_lfw_people
import numpy as np
from PIL import Image

# Load dataset with people who have multiple photos
lfw = fetch_lfw_people(min_faces_per_person=2, resize=0.5)

# Find a person with at least 2 photos
target_name = lfw.target_names[0]
indices = np.where(lfw.target == 0)[0]

print(f"Person: {target_name}")
print(f"Number of photos available: {len(indices)}")

# Save first two photos as images
img1 = Image.fromarray((lfw.images[indices[0]] * 255).astype('uint8'))
img2 = Image.fromarray((lfw.images[indices[1]] * 255).astype('uint8'))
img1.save('known_face.jpg')
img2.save('test_face.jpg')

print("Saved known_face.jpg and test_face.jpg")

Person: Aaron Peirsol
Number of photos available: 4
Saved known_face.jpg and test_face.jpg


In [ ]:
import face_recognition

known_image = face_recognition.load_image_file("known_face.jpg")
known_encoding = face_recognition.face_encodings(known_image)

test_image = face_recognition.load_image_file("test_face.jpg")
test_encoding = face_recognition.face_encodings(test_image)

if len(known_encoding) == 0 or len(test_encoding) == 0:
    print("Face not detected in one of the images")
else:
    result = face_recognition.compare_faces([known_encoding[0]], test_encoding[0])
    distance = face_recognition.face_distance([known_encoding[0]], test_encoding[0])
    print(f"Same person match: {result[0]}")
    print(f"Confidence (lower = better match): {distance[0]:.4f}")

Same person match: True
Confidence (lower = better match): 0.4765


In [ ]:
!pip install  opencv-python mediapipe


In [ ]:
!pip install mtcnn pillow -q

import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from mtcnn import MTCNN

# 1. Initialize Pure-Python Detector (No OpenCV required)
detector = MTCNN()

dataset_path = '/content/drive/MyDrive/data'

if not os.path.exists(dataset_path):
    print(f"❌ ERROR: '{dataset_path}' path nahi mila.")
else:
    image_count = 0
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_count += 1
                img_path = os.path.join(root, file)
                print(f"[{image_count}] Processing: {img_path}")

                try:
                    # Read image with PIL
                    img = Image.open(img_path).convert('RGB')

                    # Convert PIL Image to numpy array for MTCNN
                    import numpy as np
                    img_np = np.array(img)

                    # Detect Faces
                    faces = detector.detect_faces(img_np)

                    draw = ImageDraw.Draw(img)
                    detected = False

                    for face in faces:
                        x, y, w, h = face['box']
                        if w > 0 and h > 0:
                            # Draw Green Rectangle
                            draw.rectangle([x, y, x + w, y + h], outline="lime", width=5)
                            detected = True

                    if detected:
                        plt.figure(figsize=(4, 4))
                        plt.imshow(img)
                        plt.axis('off')
                        plt.title(f"Detected: {os.path.basename(root)} / {file}")
                        plt.show()
                    else:
                        print(f"⚠️ No face detected in: {file}")

                except Exception as e:
                    print(f"⚠️ Error on {file}: {e}")

In [ ]:
!pip install deepface pillow -q

import os
import pickle
import numpy as np
from PIL import Image
from deepface import DeepFace

# 1. Directories Setup
dataset_path = '/content/drive/MyDrive/data'
embeddings_save_path = 'data/student_embeddings.pkl'

known_face_encodings = []
known_face_names = []

print("🚀 Starting Face Embedding Generation...")

# 2. Iterate through all student folders
if not os.path.exists(dataset_path):
    print(f"❌ Error: Path '{dataset_path}' not found!")
else:
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(root, file)

                # Extract Student Name from Folder Name (e.g., STU_03_Priti_Renuke)
                student_name = os.path.basename(root)

                try:
                    # Generate 512-d Face Embedding using ArcFace / Facenet
                    embedding_objs = DeepFace.represent(
                        img_path=img_path,
                        model_name="ArcFace", # Options: ArcFace, Facenet, VGG-Face
                        enforce_detection=False,
                        detector_backend="opencv"
                    )

                    if embedding_objs:
                        embedding = embedding_objs[0]["embedding"]

                        known_face_encodings.append(embedding)
                        known_face_names.append(student_name)
                        print(f"✅ Encoded: {student_name} | Image: {file}")

                except Exception as e:
                    print(f"⚠️ Failed to process {file}: {e}")

    # 3. Store Embeddings safely (Auto-creates missing folders)
    if known_face_encodings:
        # Directory exist karti hai ya nahi check karke auto-create karega
        os.makedirs(os.path.dirname(embeddings_save_path), exist_ok=True)

        data_to_save = {
            "encodings": known_face_encodings,
            "names": known_face_names
        }

        with open(embeddings_save_path, "wb") as f:
            pickle.dump(data_to_save, f)

        print("\n" + "="*50)
        print(f"🎉 SUCCESS: Total {len(known_face_encodings)} face encodings generated and saved!")
        print(f"📁 Embeddings Saved At: {embeddings_save_path}")
        print("="*50)
    else:
        print("❌ No face embeddings were generated.")

🚀 Starting Face Embedding Generation...
✅ Encoded: STU_01_Prathvi_chavan | Image: image_1_01.jpeg
✅ Encoded: STU_01_Prathvi_chavan | Image: image_3_01.jpeg
✅ Encoded: STU_01_Prathvi_chavan | Image: image_2_01.jpeg
✅ Encoded: Student_09 | Image: image_02.jpg
✅ Encoded: Student_09 | Image: image_01.jpg
✅ Encoded: Student_09 | Image: image_03.jpg
✅ Encoded: Student_09 | Image: image_04.jpg
✅ Encoded: Student_09 | Image: image_05.jpg
✅ Encoded: Student_06 | Image: image_03.jpg
✅ Encoded: Student_06 | Image: image_04.jpg
✅ Encoded: Student_06 | Image: image_01.jpg
✅ Encoded: Student_06 | Image: image_02.jpg
✅ Encoded: Student_06 | Image: image_05.jpg
✅ Encoded: STU_03_Priti_Renuke | Image: STU_03IMG2.jpeg
✅ Encoded: STU_03_Priti_Renuke | Image: STU_03IMG3.jpeg
✅ Encoded: STU_03_Priti_Renuke | Image: STU_03_IMG1.jpeg
✅ Encoded: Student_05 | Image: image_04.jpg
✅ Encoded: Student_05 | Image: image_02.jpg
✅ Encoded: Student_05 | Image: image_03.jpg
✅ Encoded: Student_05 | Image: image_05.jpg
✅

In [14]:
import pickle

# Pickle file ko read (unpickle) karein
with open('data/student_embeddings.pkl', 'rb') as f:
    data = pickle.load(f)

print("✅ File successfully loaded!")
print(f"Total Encodings Saved: {len(data['encodings'])}")
print("\nSaved Student Names:")
print(set(data['names']))  # Unique student names display karega

✅ File successfully loaded!
Total Encodings Saved: 37

Saved Student Names:
{'STU_01_Prathvi_chavan', 'Student_08', 'Student_05', 'Student_07', 'Student_06', 'STU_02_Akshata_Arsul', 'Student_09', 'STU_04_Neeraj', 'STU_03_Priti_Renuke'}


In [18]:
import os
import pickle
import numpy as np
import pandas as pd
from datetime import datetime
from deepface import DeepFace

# 1. Paths Setup
embeddings_path = '/content/data/student_embeddings.pkl'
attendance_file = 'data/attendance.csv'
dataset_path = '/content/drive/MyDrive/data'

# 2. Load Stored Embeddings Database
print("📂 Loading stored embeddings database...")
with open(embeddings_path, 'rb') as f:
    database = pickle.load(f)

known_encodings = np.array(database['encodings'])
known_names = database['names']

# 3. Vector Distance Matching Function
def verify_student(test_vector, threshold=0.45):
    distances = np.linalg.norm(known_encodings - test_vector, axis=1)
    best_match_index = np.argmin(distances)
    min_distance = distances[best_match_index]

    if min_distance <= threshold:
        return known_names[best_match_index], min_distance
    return "Unknown", min_distance

# 4. Find Any Available Image Automatically for Testing
test_image_path = None
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            test_image_path = os.path.join(root, file)
            break
    if test_image_path:
        break

if not test_image_path:
    print("❌ Dataset me koi test image nahi mili!")
else:
    print(f"🔍 Testing on automatically picked image: {test_image_path}")

    try:
        # Generate embedding for test image
        representation = DeepFace.represent(
            img_path=test_image_path,
            model_name="ArcFace",
            enforce_detection=False,
            detector_backend="opencv"
        )
        test_embedding = representation[0]["embedding"]

        # Run Face Matching
        matched_name, match_score = verify_student(test_embedding)

        print("\n" + "="*50)
        print(f"🎯 RECOGNITION RESULT:")
        print(f"👤 Identified Student : {matched_name}")
        print(f"📏 Distance Score    : {match_score:.4f}")
        print("="*50)

        # 5. Mark Attendance in CSV
        if matched_name != "Unknown":
            now = datetime.now()
            date_today = now.strftime("%Y-%m-%d")
            time_now = now.strftime("%H:%M:%S")

            new_entry = pd.DataFrame([{
                'Student_Name': matched_name,
                'Date': date_today,
                'Time': time_now,
                'Status': 'Present'
            }])

            if os.path.exists(attendance_file):
                df_existing = pd.read_csv(attendance_file)
                dup = df_existing[(df_existing['Student_Name'] == matched_name) & (df_existing['Date'] == date_today)]
                if dup.empty:
                    df_final = pd.concat([df_existing, new_entry], ignore_index=True)
                    df_final.to_csv(attendance_file, index=False)
                    print(f"✅ Attendance Marked for {matched_name}")
                else:
                    print(f"⚠️ Attendance already marked today for {matched_name}.")
            else:
                new_entry.to_csv(attendance_file, index=False)
                print(f"✅ Created '{attendance_file}' & marked attendance for {matched_name}.")

    except Exception as e:
        print(f"❌ Matching Error: {e}")

📂 Loading stored embeddings database...
🔍 Testing on automatically picked image: /content/drive/MyDrive/data/raw_student_dataset/STU_01_Prathvi_chavan/image_1_01.jpeg

🎯 RECOGNITION RESULT:
👤 Identified Student : STU_01_Prathvi_chavan
📏 Distance Score    : 0.0000
✅ Created 'data/attendance.csv' & marked attendance for STU_01_Prathvi_chavan.


In [19]:
import os
import pickle
import numpy as np
import pandas as pd
from datetime import datetime
from deepface import DeepFace

# 1. Drive Paths Setup
embeddings_path = '/content/data/student_embeddings.pkl'
attendance_file = '/content/drive/MyDrive/data/attendance.csv'
dataset_path = '//content/drive/MyDrive/data'

# Fallback path if files are in local Colab root
if not os.path.exists(embeddings_path) and os.path.exists('data/student_embeddings.pkl'):
    embeddings_path = 'data/student_embeddings.pkl'
    attendance_file = 'data/attendance.csv'
    dataset_path = 'data/raw_student_dataset'

# 2. Load Stored Embeddings Database
print("📂 Loading stored embeddings database...")
with open(embeddings_path, 'rb') as f:
    database = pickle.load(f)

known_encodings = np.array(database['encodings'])
known_names = database['names']

# 3. Vector Distance Matching Function
def verify_student(test_vector, threshold=0.45):
    distances = np.linalg.norm(known_encodings - test_vector, axis=1)
    best_match_index = np.argmin(distances)
    min_distance = distances[best_match_index]

    if min_distance <= threshold:
        return known_names[best_match_index], min_distance
    return "Unknown", min_distance

# 4. Find Any Available Image Automatically in Drive
test_image_path = None
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            test_image_path = os.path.join(root, file)
            break
    if test_image_path:
        break

if not test_image_path:
    print(f"❌ '{dataset_path}' me koi image nahi mili! Path check karein.")
else:
    print(f"🔍 Testing on automatically picked image: {test_image_path}")

    try:
        representation = DeepFace.represent(
            img_path=test_image_path,
            model_name="ArcFace",
            enforce_detection=False,
            detector_backend="opencv"
        )
        test_embedding = representation[0]["embedding"]

        # Face Matching
        matched_name, match_score = verify_student(test_embedding)

        print("\n" + "="*50)
        print(f"🎯 RECOGNITION RESULT:")
        print(f"👤 Identified Student : {matched_name}")
        print(f"📏 Distance Score    : {match_score:.4f}")
        print("="*50)

        # Mark Attendance
        if matched_name != "Unknown":
            now = datetime.now()
            date_today = now.strftime("%Y-%m-%d")
            time_now = now.strftime("%H:%M:%S")

            new_entry = pd.DataFrame([{
                'Student_Name': matched_name,
                'Date': date_today,
                'Time': time_now,
                'Status': 'Present'
            }])

            if os.path.exists(attendance_file):
                df_existing = pd.read_csv(attendance_file)
                dup = df_existing[(df_existing['Student_Name'] == matched_name) & (df_existing['Date'] == date_today)]
                if dup.empty:
                    df_final = pd.concat([df_existing, new_entry], ignore_index=True)
                    df_final.to_csv(attendance_file, index=False)
                    print(f"✅ Attendance Marked for {matched_name}")
                else:
                    print(f"⚠️ Attendance already marked today for {matched_name}.")
            else:
                new_entry.to_csv(attendance_file, index=False)
                print(f"✅ Created '{attendance_file}' & marked attendance for {matched_name}.")

    except Exception as e:
        print(f"❌ Matching Error: {e}")

📂 Loading stored embeddings database...
🔍 Testing on automatically picked image: //content/drive/MyDrive/data/raw_student_dataset/STU_01_Prathvi_chavan/image_1_01.jpeg

🎯 RECOGNITION RESULT:
👤 Identified Student : STU_01_Prathvi_chavan
📏 Distance Score    : 0.0000
✅ Created '/content/drive/MyDrive/data/attendance.csv' & marked attendance for STU_01_Prathvi_chavan.
